# Perception-Limited Scenario Generation — Result Analysis & Visualization

Merge the dangerous-scenario libraries generated across multiple seeds, output statistics, distributions, and visualizations to analyze the overall distribution of all generated dangerous scenarios.

Data source: `ppo_logs_gpu*/vae-ppo_vehicle_trajectories_*.h5` (one independent seed per GPU).

## Data schema

- Each `episode_*` group, attrs: `collision / episode_length / episode_reward`
  - `trajectories` `(T, 5, 4)`: `[longitudinal position, lateral position lane_pos, longitudinal speed, 0]`, vehicle order `[ego, adversary, bg2, bg3, bg4]`
  - `perception_data` `(T, 5, 13)`: `[perceived(4) | true(4) | error delta(4) | distance(1)]`
    - perceived: `[perceived_pos, perceived_lane_pos, perceived_speed, perceived_vy]`
    - true: `[true_pos, true_lane_pos, true_speed, true_vy]`
    - error: `[dx, dy, dvx, dvy]`
    - distance: `dist_to_ego`

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import h5py
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['axes.grid'] = True

# ===== Configuration (edit as needed) =====
DATA_DIR = "/workspace/asw-shared/dlp/training_tasks/aoh6szh/n260827-174817-scegen-ppo"
H5_GLOB = os.path.join(DATA_DIR, "ppo_logs_gpu*/vae-ppo_vehicle_trajectories_*.h5")

# ===== Constants (must match sce_gen.py) =====
CAR_LENGTH = 5.0
LANE_WIDTH = 4.0
SAME_LANE_THRESH = 2.2
TTC_LOW = 1.0
TTC_HIGH = 4.0
TIME_STEP = 0.2

# Fitted perception-noise coefficients (paper Table III), used to overlay the fit curves.
PERC_COEFFS = {
    'dx':  {'mu': (-0.00691, -0.00013, -4.64218e-07), 'sigma': (0.03711, 0.00511, -5.68800e-05)},
    'dy':  {'mu': (-0.00571,  0.00039, -4.90203e-08), 'sigma': (0.06703, 0.00593, -7.43626e-05)},
    'dvx': {'mu': (-0.00369,  0.00054, -6.30793e-06), 'sigma': (0.06579, 0.00693, -7.20104e-05)},
    'dvy': {'mu': ( 0.00864, -0.00323,  6.23654e-05), 'sigma': (0.31881, 0.00069,  5.11978e-06)},
}

files = sorted(glob.glob(H5_GLOB))
print(f"found {len(files)} h5 files")
for f in files:
    print("  ", f)

In [ ]:
def min_ttc_of_trajectory(traj):
    """traj: (T, N, 4) = [pos, lane_pos, speed, 0]; car0 = ego. Return the minimum TTC of this episode."""
    ego = traj[:, 0, :]
    best = np.inf
    with np.errstate(divide='ignore', invalid='ignore'):
        for i in range(1, traj.shape[1]):
            npc = traj[:, i, :]
            dx = npc[:, 0] - ego[:, 0]
            rel = ego[:, 2] - npc[:, 2]
            lane_ok = np.abs(ego[:, 1] - npc[:, 1]) < SAME_LANE_THRESH
            for m, ttc in ((lane_ok & (dx > 0) & (rel > 0), (dx - CAR_LENGTH) / rel),
                           (lane_ok & (dx < 0) & (rel < 0), (-dx - CAR_LENGTH) / (-rel))):
                if m.any():
                    v = ttc[m]
                    v = v[v > 0]
                    if v.size:
                        best = min(best, v.min())
    return best if np.isfinite(best) else 100.0


def poly2(c, d):
    a0, a1, a2 = c
    return a0 + a1 * d + a2 * d ** 2


def ttc_matrix(ego, others, perceived):
    """Compute, per timestep, the minimum TTC of the ego against all background vehicles.

    ego: (T, 1, 13); others: (T, M, 13).
    perceived=True uses the perceived columns (0, 1, 2), False uses the true columns (4, 5, 6). Returns a (T,) array.
    """
    pos, lane, spd = (0, 1, 2) if perceived else (4, 5, 6)
    ego_pos = ego[:, :, pos]
    ego_lane = ego[:, :, lane]
    ego_spd = ego[:, :, spd]
    o_pos = others[:, :, pos]
    o_lane = others[:, :, lane]
    o_spd = others[:, :, spd]
    dx = o_pos - ego_pos
    rel = ego_spd - o_spd
    lane_ok = np.abs(ego_lane - o_lane) < SAME_LANE_THRESH
    with np.errstate(divide='ignore', invalid='ignore'):
        fwd = (dx - CAR_LENGTH) / rel
        rear = (-dx - CAR_LENGTH) / (-rel)
    fwd = np.where(lane_ok & (dx > 0) & (rel > 0) & (fwd > 0), fwd, np.inf)
    rear = np.where(lane_ok & (dx < 0) & (rel < 0) & (rear > 0), rear, np.inf)
    return np.minimum(fwd, rear).min(axis=1)

In [ ]:
rows = []
for path in files:
    run = os.path.basename(os.path.dirname(path))  # ppo_logs_gpu0_...
    with h5py.File(path, 'r') as f:
        for k in f.keys():
            g = f[k]
            attrs = dict(g.attrs)
            traj = g['trajectories'][:]  # (T,5,4)
            rows.append(dict(
                run=run,
                episode=int(k.split('_')[1]),
                length=traj.shape[0],
                collision=bool(attrs.get('collision', False)),
                reward=float(attrs.get('episode_reward', 0.0)),
                min_ttc=min_ttc_of_trajectory(traj),
                path=path,
            ))
meta = pd.DataFrame(rows)
print("total scenarios:", len(meta))
meta.head()

In [ ]:
print("===== Overall statistics =====")
print(f"total scenarios       : {len(meta)}")
print(f"num runs (seeds)      : {meta['run'].nunique()}")
print(f"collisions            : {int(meta['collision'].sum())}  ({100*meta['collision'].mean():.2f}%)")
print(f"dangerous TTC<{TTC_HIGH:.0f}: {(meta['min_ttc']<TTC_HIGH).sum()}  ({100*(meta['min_ttc']<TTC_HIGH).mean():.2f}%)")
print(f"severe TTC<{TTC_LOW:.0f}: {(meta['min_ttc']<TTC_LOW).sum()}  ({100*(meta['min_ttc']<TTC_LOW).mean():.2f}%)")
print()
print(meta[['length', 'reward', 'min_ttc']].describe().round(3))

===== Overall statistics =====
total scenarios       : 9988
num runs (seeds)      : 4
collisions            : 306  (3.06%)
dangerous TTC<4: 1189  (11.90%)
severe TTC<1: 454  (4.55%)

         length    reward     min_ttc
count  9988.000  9988.000    9988.000
mean     80.378     4.129     138.579
std      63.810    31.434    7680.422
min       4.000    -5.000       0.001
25%      27.000    -5.000       8.214
50%      61.000    -4.885      20.595
75%     124.000     0.000     100.000
max     212.000   489.394  766605.625

In [ ]:
by_run = meta.groupby('run').agg(
    episodes=('episode', 'count'),
    collision_rate=('collision', 'mean'),
    dangerous_rate=('min_ttc', lambda s: (s < TTC_HIGH).mean()),
    mean_min_ttc=('min_ttc', 'mean'),
    mean_length=('length', 'mean'),
    mean_reward=('reward', 'mean'),
).reset_index()
by_run.round(4)


run	episodes	collision_rate	dangerous_rate	mean_min_ttc	mean_length	mean_reward
0	ppo_logs_gpu0_20260827_175515	2552	0.0357	0.1101	67.1735	78.6458	3.4801
1	ppo_logs_gpu1_20260827_175515	2595	0.0258	0.1214	64.8782	77.3426	3.9579
2	ppo_logs_gpu2_20260827_175515	2349	0.0307	0.1222	55.1646	85.4423	4.9127
3	ppo_logs_gpu3_20260827_175515	2492	0.0305	0.1228	367.0785	80.5393	4.2312


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].hist(meta['length'], bins=40, color='tab:blue', alpha=0.7)
axes[0, 0].set_title('Episode length (steps)')
axes[0, 0].set_xlabel('steps')

ttc = meta['min_ttc'].clip(lower=0.05)
axes[0, 1].hist(ttc, bins=60, color='tab:orange', alpha=0.7)
axes[0, 1].set_xscale('log')
axes[0, 1].axvline(TTC_LOW, color='red', ls='--', label='TTC_LOW=1')
axes[0, 1].axvline(TTC_HIGH, color='green', ls='--', label='TTC_HIGH=4')
axes[0, 1].set_title('Min TTC (log scale)')
axes[0, 1].legend()

axes[0, 2].hist(meta['reward'], bins=50, color='tab:green', alpha=0.7)
axes[0, 2].set_title('Episode reward')

axes[1, 0].bar(by_run['run'], by_run['collision_rate'] * 100, color='tab:red', alpha=0.7)
axes[1, 0].set_title('Collision rate per run (%)')
axes[1, 0].tick_params(axis='x', rotation=45)

axes[1, 1].bar(by_run['run'], by_run['dangerous_rate'] * 100, color='tab:purple', alpha=0.7)
axes[1, 1].set_title(f'Dangerous rate (TTC<{TTC_HIGH:.0f}) per run (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

axes[1, 2].boxplot([meta[meta['run'] == r]['min_ttc'].clip(lower=0.05) for r in by_run['run']],
                   labels=by_run['run'])
axes[1, 2].set_yscale('log')
axes[1, 2].set_title('Min TTC per run (log)')
axes[1, 2].tick_params(axis='x', rotation=45)

fig.tight_layout()
plt.show()

## Perception-error analysis

Extract `(dist, dx, dy, dvx, dvy)` for every non-ego vehicle across all timesteps from `perception_data`, analyze the error distribution and how the error changes with distance, and overlay the fitted `mu(d)` / `sigma(d)` curves from the paper for verification.

In [ ]:
def collect_perception_errors(files):
    dists, dxs, dys, dvxs, dvys = [], [], [], [], []
    for path in files:
        with h5py.File(path, 'r') as f:
            for k in f.keys():
                p = f[k]['perception_data'][:]  # (T,5,13)
                npe = p[:, 1:, :]  # non-ego vehicles
                dists.append(npe[:, :, 12].ravel())
                dxs.append(npe[:, :, 8].ravel())
                dys.append(npe[:, :, 9].ravel())
                dvxs.append(npe[:, :, 10].ravel())
                dvys.append(npe[:, :, 11].ravel())
    return tuple(np.concatenate(x) for x in (dists, dxs, dys, dvxs, dvys))

dist, dx, dy, dvx, dvy = collect_perception_errors(files)
print(f"perception-error samples: {len(dist):,}")
print(pd.DataFrame({'dx': dx, 'dy': dy, 'dvx': dvx, 'dvy': dvy}).describe().round(4))

perception-error samples: 3,211,264
                 dx            dy           dvx           dvy
count  3.211264e+06  3.211264e+06  3.211264e+06  3.211264e+06
mean  -2.600000e-02  3.070000e-02 -4.400000e-02  6.069000e-01
std    9.370000e-02  1.115000e-01  2.175000e-01  1.800000e+00
min   -7.163000e-01 -9.273000e-01 -7.096800e+00 -1.915100e+00
25%   -5.580000e-02  4.200000e-03 -1.096000e-01 -7.900000e-02
50%   -3.150000e-02  4.130000e-02 -3.850000e-02  2.585000e-01
75%    6.000000e-04  6.840000e-02  3.750000e-02  7.507000e-01
max    7.059000e-01  9.061000e-01  1.155200e+00  8.303100e+01

In [ ]:
errors = {'dx': dx, 'dy': dy, 'dvx': dvx, 'dvy': dvy}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (name, e) in zip(axes.ravel(), errors.items()):
    ax.hist(e, bins=80, density=True, alpha=0.7, color='tab:blue')
    ax.set_title(f'{name} error distribution')
    ax.set_xlabel(name)
    ax.set_ylabel('density')
fig.tight_layout()
plt.show()

# Error vs distance + overlay mu(d), mu +/- sigma(d)
d_grid = np.linspace(0, dist.max(), 200)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (name, e) in zip(axes.ravel(), errors.items()):
    mu_c = PERC_COEFFS[name]['mu']
    sg_c = PERC_COEFFS[name]['sigma']
    ax.hexbin(dist, e, gridsize=60, bins='log', cmap='Blues', mincnt=1)
    mu_d = poly2(mu_c, d_grid)
    sg_d = np.abs(poly2(sg_c, d_grid))
    ax.plot(d_grid, mu_d, 'r-', lw=2, label='mu(d)')
    ax.plot(d_grid, mu_d + sg_d, 'r--', lw=1, label='mu +/- sigma(d)')
    ax.plot(d_grid, mu_d - sg_d, 'r--', lw=1)
    ax.set_xlabel('distance d (m)')
    ax.set_ylabel(name)
    ax.set_title(f'{name} vs distance')
    ax.legend()
fig.tight_layout()
plt.show()

## Perception-induced risk underestimation (ERD-lite)

The paper's ERD computes `Delta = E[R_obs] - E[R_true]` using forward IDM simulation. Here we approximate it with the instantaneous per-timestep TTC: compute the perceived TTC from the ego's perceived view and the true TTC from the true view, with `risk = 1/TTC`. `risk_true - risk_perc > 0` means the ego **underestimates the danger** (believes it is safer than it actually is).

In [ ]:
def collect_risk_underestimation(files):
    all_true, all_perc, all_under = [], [], []
    for path in files:
        with h5py.File(path, 'r') as f:
            for k in f.keys():
                p = f[k]['perception_data'][:]
                ego = p[:, 0:1, :]
                others = p[:, 1:, :]
                t_true = ttc_matrix(ego, others, perceived=False)
                t_perc = ttc_matrix(ego, others, perceived=True)
                r_true = 1.0 / np.clip(t_true, 0.1, None)
                r_perc = 1.0 / np.clip(t_perc, 0.1, None)
                all_true.append(t_true)
                all_perc.append(t_perc)
                all_under.append(r_true - r_perc)
    return (np.concatenate(x) for x in (all_true, all_perc, all_under))

ttc_true, ttc_perc, under = collect_risk_underestimation(files)
print(f"timestep samples: {len(ttc_true):,}")

danger = ttc_true < TTC_HIGH
print(f"\ntrue TTC<{TTC_HIGH:.0f} timestep fraction: {100*danger.mean():.2f}%")
print(f"  among them, perceived TTC > true TTC (danger underestimated): {100*(ttc_perc[danger] > ttc_true[danger]).mean():.2f}%")
print(f"  mean underestimation magnitude: {under[danger & (under > 0)].mean():.3f} (1/TTC units)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(under, bins=80, color='tab:red', alpha=0.7)
axes[0].axvline(0, color='k', ls='--')
axes[0].set_title('Risk underestimation = risk_true - risk_perc (positive = danger underestimated)')
axes[0].set_xlabel('risk_true - risk_perc')

m = (ttc_true < 20) & (ttc_perc < 20)
axes[1].hexbin(ttc_true[m], ttc_perc[m], gridsize=60, bins='log', cmap='Greens', mincnt=1)
axes[1].plot([0, 20], [0, 20], 'r--', label='perceived = true')
axes[1].set_xlabel('true TTC (s)')
axes[1].set_ylabel('perceived TTC (s)')
axes[1].set_title('Perceived vs true TTC (above diagonal = danger underestimated)')
axes[1].legend()
plt.tight_layout()
plt.show()

timestep samples: 802,816

true TTC<4 timestep fraction: 0.39%
  among them, perceived TTC > true TTC (danger underestimated): 54.17%
  mean underestimation magnitude: 0.066 (1/TTC units)

## Sample-scenario visualization

Pick one collision scenario and one minimum-TTC scenario, and plot the longitudinal and lateral positions of each vehicle over time.

In [ ]:
def plot_episode(path, episode_idx):
    with h5py.File(path, 'r') as f:
        g = f[f"episode_{episode_idx}"]
        traj = g['trajectories'][:]
        attrs = dict(g.attrs)
    T = traj.shape[0]
    t = np.arange(T) * TIME_STEP
    names = ['ego', 'adversary', 'bg2', 'bg3', 'bg4']
    colors = ['black', 'red', 'tab:gray', 'tab:gray', 'tab:gray']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for i in range(traj.shape[1]):
        axes[0].plot(t, traj[:, i, 0], color=colors[i], label=names[i])
        axes[1].plot(t, traj[:, i, 1], color=colors[i], label=names[i])
    axes[0].set_xlabel('time (s)'); axes[0].set_ylabel('longitudinal position (m)')
    axes[0].set_title(f"episode_{episode_idx} longitudinal position (collision={attrs.get('collision')})")
    axes[1].set_xlabel('time (s)'); axes[1].set_ylabel('lateral position lane_pos (m)')
    axes[1].set_title('lateral position / lane change')
    for ax in axes:
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

coll = meta[meta['collision']]
if len(coll):
    r = coll.iloc[0]
    plot_episode(r['path'], r['episode'])

near = meta[meta['min_ttc'] == meta['min_ttc'].min()]
if len(near):
    r = near.iloc[0]
    plot_episode(r['path'], r['episode'])

## Pick top-10 representative perception-limited dangerous scenarios

Goal: from ~10k episodes, shortlist 10 that best illustrate "ego underestimates danger because of perception noise", then inspect each one visually and keep the best for the paper figure.

**Scoring** (improvements over the single-frame `select_episode` in `visualize_scenario.py`):

1. **Candidate pool**: `collision` or `min_ttc < TTC_HIGH (4s)` — must be genuinely dangerous.
2. **Score** blends four capped signals so one extreme episode can't dominate:
   - `max(0, TTC_LOW - min_ttc)` — danger severity
   - `0.5 * min(under_at_danger, 10)` — perception underestimate at the worst frame (the core perception-limited signal)
   - `0.5 * min(mean_under_window, 3)` — sustained underestimation across the whole danger window (rewards real misjudgement, not a one-frame fluke)
   - `0.05 * danger_window_s` — bonus for sustained proximity
3. **Why these additions**: the original `visualize_scenario.py` only looked at `argmin(ttc_true)` — a single frame. A collision episode often has `ttc_perc = cap = 100` at the impact frame, so the "underestimate" signal there is artificial. Looking at the *mean across the danger window* catches episodes where the ego was *persistently* misreading the situation, which is more representative of the perception-limited failure mode we want to illustrate.

**Usage**: run the scoring cell, then walk through `top10` with the inspector cell below it by changing `i = 0..9` and re-running. Each call shows longitudinal + lateral positions of all vehicles for that episode.

In [ ]:
# Pick the top-10 "most representative" perception-limited dangerous scenarios.
#
# Ranking criteria (improved over the original single-frame score):
#   1. Must be dangerous: min_ttc < TTC_HIGH (4s); collisions preferred.
#   2. Score blends three signals so we don't pick a one-frame fluke:
#      a) Danger severity:        (TTC_LOW - min_ttc)         — cap with TTC_LOW
#      b) Underestimation at the danger frame (core perception-limited signal)
#      c) Mean underestimation across the whole danger window — rewards sustained misjudgement
#      d) Danger-window length bonus                             — rewards sustained proximity
#   3. Cap the score components so a single extreme episode doesn't dominate.

TTC_CAP = 100.0
UNDER_SINGLE_CAP = 10.0   # cap on per-frame underestimation contribution (s)
UNDER_MEAN_CAP   = 3.0    # cap on mean-window underestimation contribution (s)
WINDOW_BONUS_W   = 0.05   # weight for danger-window duration (per second)

def score_episode(path, key, traj, perc_full, collision):
    """traj: (T,5,4) true; perc_full: (T,5,13). Returns a score dict."""
    ego_p = perc_full[:, 0:1, :]
    oth_p = perc_full[:, 1:, :]
    t_true = ttc_matrix(ego_p, oth_p, perceived=False)
    t_perc = ttc_matrix(ego_p, oth_p, perceived=True)
    t_true = np.where(np.isinf(t_true), TTC_CAP, t_true)
    t_perc = np.where(np.isinf(t_perc), TTC_CAP, t_perc)

    danger_mask = t_true < TTC_HIGH
    danger_window_s = float(danger_mask.sum()) * TIME_STEP

    min_ttc = float(t_true.min())
    i = int(np.argmin(t_true))
    under_at = float(max(0.0, t_perc[i] - t_true[i]))

    if danger_mask.any():
        under_window = np.maximum(0.0, t_perc[danger_mask] - t_true[danger_mask])
        under_mean = float(under_window.mean())
    else:
        under_mean = 0.0

    score = (
        max(0.0, TTC_LOW - min_ttc)
        + 0.5 * min(under_at, UNDER_SINGLE_CAP)
        + 0.5 * min(under_mean, UNDER_MEAN_CAP)
        + WINDOW_BONUS_W * danger_window_s
    )
    return dict(
        min_ttc=min_ttc, min_ttc_frame=i,
        under_at_danger=under_at, mean_under_window=under_mean,
        danger_window_s=danger_window_s, score=score,
    )

# Build candidate pool: dangerous + collision episodes.
cand = meta[meta['collision'] | (meta['min_ttc'] < TTC_HIGH)].copy()
print(f"candidate pool: {len(cand)} episodes (collisions + TTC<{TTC_HIGH:.0f})")

rows = []
for _, r in cand.iterrows():
    with h5py.File(r['path'], 'r') as f:
        g = f[f"episode_{r['episode']}"]
        traj = g['trajectories'][:]
        perc_full = g['perception_data'][:]
    s = score_episode(r['path'], f"episode_{r['episode']}", traj, perc_full,
                      bool(r['collision']))
    s.update(run=r['run'], episode=int(r['episode']),
             collision=bool(r['collision']), path=r['path'])
    rows.append(s)

ranked = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
top10 = ranked.head(10).copy()

print("\n===== Top-10 representative perception-limited dangerous scenarios =====")
print(top10[['run', 'episode', 'collision', 'min_ttc', 'min_ttc_frame',
            'under_at_danger', 'mean_under_window', 'danger_window_s', 'score']]
      .to_string(index=False))

In [ ]:
# Inspect each candidate in turn — change i to step through them.
# plot_episode (defined above) shows longitudinal + lateral position over time for all vehicles.
i = 0
r = top10.iloc[i]
print(f"[rank {i}] run={r['run']}  episode={r['episode']}  collision={r['collision']}")
print(f"        min_ttc={r['min_ttc']:.2f}s @ frame {r['min_ttc_frame']}  "
      f"under_at_danger={r['under_at_danger']:.2f}s")
print(f"        mean_under_window={r['mean_under_window']:.2f}s  "
      f"danger_window={r['danger_window_s']:.1f}s  score={r['score']:.2f}")
plot_episode(r['path'], r['episode'])